# Coastal — consensus workflow (post-audit)

The clean, current end-to-end path, built from the real steps in `optical_flow.ipynb` and
`pipeline_confetti_ceiling.ipynb` with removed dead ends stripped out. Those notebooks keep the
full experiment history and reference approaches no longer in the package — see
[`../docs/DEAD_ENDS.md`](../docs/DEAD_ENDS.md).

**Train on movies → run the pipeline on held-out movies.** We fit the flow-metric UNet on a set of
TRAIN movies, then run segmentation → tracking → scoring on separate TEST movies the model never
saw. (This avoids the leakage of a per-frame split, where you'd segment whole movies whose early
frames trained the model.)

**Pipeline:** cecelia loads `[T,C,Z,Y,X]` → flow-metric UNet (no labels, 3 losses) → 3D+T instance
labels (`Inference3D`, IOU-stitched across Z) → Kalman + LAP tracking (Mahalanobis + `w_flow` +
`w_color`) → scoring vs confetti identity. cecelia does I/O; coastal takes the array in, returns
labels out (see [`../docs/DATA.md`](../docs/DATA.md)).

**Kernel:** **Python (coastal)** (`pixi run kernel`, then `pixi run lab`). Device auto-selects
(cuda→mps→cpu) — no `device=` needed unless overriding.

In [ ]:
import os
import gc
import numpy as np

from coastal import (
    prepare_data_for_unet_batch_4d,   # 4D volumes -> 2D training sequences (flow + variance metrics)
    train_with_metrics, save_model, load_model,
    LearnedAffinityInference, Inference3D,
    optimize_segmentation_cma, score_segmentation,
    filter_small_cells,
    compute_cell_flow_features, extract_cell_intensities,
    track_sequence, score_tracking,
)

# cecelia IO helpers (installed editable in the pixi env). See docs/DATA.md.
import cecelia.utils.zarr_utils as zarr_utils
import cecelia.utils.ome_xml_utils as ome_xml_utils
from cecelia.utils.dim_utils import DimUtils

print('ready')

## Config

`CH_INDICES_MAP` gives the confetti channels **per movie** (acquisitions differ). Training uses one
channel set (`TRAIN_CH`), so keep TRAIN/TEST movies within the same channel group. `BEST_PARAMS`
are the tuned segmentation-inference params (from `optimize_segmentation_cma`).

In [ ]:
BASE_DIR = os.environ.get('COASTAL_BASE_DIR', '/home/dominik/cecelia/projects/R0YiZv/ANALYSIS')
ZARR_NAME = 'ccidDriftCorrected.zarr'

# Per-movie confetti channels.
CH_INDICES_MAP = {
    **{u: [1, 2, 3] for u in ['QS0Z6B', 'E25ivV', 'LlBxY9', 'iRxilR', '2CHUML',
                              'BVnKB3', 'ELoL5a', 'fFnZOv', 'CdQYH0']},
    **{u: [0, 2, 3] for u in ['hfMMQ1', 'IRaTvy', 'rK8nQY', 'AksHak', '4D4QUc',
                              'B1OSJo', 'mI9UUN']},
}

# Movie-level split (same channel group so TRAIN_CH is valid for the train set).
TEST_UIDS  = ['fFnZOv', 'CdQYH0']                                   # held out — never seen in training
TRAIN_UIDS = ['QS0Z6B', 'E25ivV', 'LlBxY9', 'iRxilR', '2CHUML', 'BVnKB3', 'ELoL5a']
TRAIN_CH   = CH_INDICES_MAP[TRAIN_UIDS[0]]                          # channels for training-flow metrics

MIN_VOXELS  = 200
TARGET_SIZE = (384, 384)

BEST_PARAMS = {
    'seed_size': 12, 'embedding_blur_sigma': 1.0, 'prob_threshold': 0.4,
    'min_component_size': 12, 'max_iter': 200, 'min_boundary_pixels': 8,
    'affinity_threshold': 0.5534, 'merge_affinity_threshold': 0.2261,
    'merge_max_distance': 0.6198, 'merge_contact_brightness_threshold': 0.3733,
    'prob_weight': 0.3366,
}
print('train', TRAIN_UIDS, '| test', TEST_UIDS)

## 1. Load movies (cecelia — the array-boundary seam)

Keep each image as its cecelia dask pyramid (`volumes[uid]`); `vol[0]` is full-res. `pix_res` from
`DimUtils`. We load both the train and test movies here.

In [ ]:
volumes = {}
pix_res = {}
for uid in TRAIN_UIDS + TEST_UIDS:
    im_path = os.path.join(BASE_DIR, f'0/{uid}/{ZARR_NAME}')
    im, _ = zarr_utils.open_as_zarr(im_path, as_dask=True)          # pyramid list; vol[0] = full-res
    dim_utils = DimUtils(ome_xml_utils.parse_meta(im_path), use_channel_axis=True)
    dim_utils.calc_image_dimensions(im[0].shape)
    volumes[uid] = im
    pix_res[uid] = dim_utils.im_physical_sizes()                    # {'z':..,'y':..,'x':..} µm/px
    print(uid, np.array(im[0]).shape, pix_res[uid])

## 2. Training sequences from the TRAIN movies (no ground-truth labels)

`prepare_data_for_unet_batch_4d` samples 2D `[T,H,W]` sequences (evenly-spaced Z-slices, random
start times), computing flow metrics on the **mean-projected** channel and variance metrics on the
full multi-channel data. Only TRAIN movies go in. Returns a 4-tuple; flatten for training.

In [ ]:
train_volumes = {u: volumes[u] for u in TRAIN_UIDS}

all_frames, all_temporal, all_variance, all_frames_multi = prepare_data_for_unet_batch_4d(
    train_volumes,
    n_sequences=3,          # z-slices sampled per volume
    seq_len=20,             # frames per sequence
    ch_indices=TRAIN_CH,
    temporal_scales=[1, 2, 4],
    cumulative_window=2,
    random_seed=42,
    target_size=TARGET_SIZE,
)

train_frames   = np.concatenate(all_frames, axis=0)         # [N, H, W]
train_temporal = [m for seq in all_temporal for m in seq]   # flat list of dicts
train_variance = [m for seq in all_variance for m in seq]

# One TRAIN sequence held only for a quick visual / tuning check (not a test movie).
eval_frames, eval_temporal, eval_frames_multi = all_frames[0], all_temporal[0], all_frames_multi[0]
print('train frames', train_frames.shape, '| sequences', len(all_frames))

## 3. Train the UNet (intensity + temporal + variance losses)

Self-supervised on flow structure. `device` auto-resolves. Run once, `save_model`, reload later.

In [ ]:
model, history = train_with_metrics(
    train_frames,
    train_temporal,
    variance_metrics_norm=train_variance,
    num_epochs=80,
    batch_size=4,
    embedding_dim=64,
    intensity_weight=1.0,
    temporal_weight=1.0,
    variance_weight=1.0,
    variance_window_size=32,
    variance_dropout_p=0.5,
)

In [ ]:
MODEL_PATH = os.environ.get('COASTAL_MODEL', 'coastal_model.pt')
# save_model(model, MODEL_PATH, metadata={'num_epochs': 80, 'target_size': TARGET_SIZE})
# Later runs: model = load_model(MODEL_PATH)
#
# (Optional) tune BEST_PARAMS on a TRAIN eval sequence with CMA-ES:
# best_params, _ = optimize_segmentation_cma(
#     model=model, frames=eval_frames, frames_multi=eval_frames_multi, temporal_metrics=eval_temporal,
#     x0=[0.5, 0.5, 1.0, 0.5, 0.5], sigma0=0.20, max_evals=20,
#     n_frames=5, min_cell_size=100, purity_threshold=0.8,
#     fixed_params={'seed_size': 12, 'embedding_blur_sigma': 1.0, 'prob_threshold': 0.4,
#                   'min_component_size': 12, 'max_iter': 200, 'min_boundary_pixels': 8})

## 4. Segment the HELD-OUT TEST movies → instance labels

The model never saw these movies. `Inference3D` runs per Z-slice + IOU-stitches across Z;
`predict_temporal_volume` returns a **`(instances, results)` tuple**. Uses each movie's own channels.

In [ ]:
inferencer_3d = Inference3D(model=model, **BEST_PARAMS)

instances_per_uid = {}
for uid in TEST_UIDS:
    inst, _ = inferencer_3d.predict_temporal_volume(
        np.asarray(volumes[uid][0]), ch_indices=CH_INDICES_MAP[uid],
        stitch_threshold=0.1, gap_tolerance=1, gap_iou_threshold=0.3, n_workers=8)
    instances_per_uid[uid] = filter_small_cells(inst, min_voxels=MIN_VOXELS)
    n = np.mean([len(np.unique(instances_per_uid[uid][t])) - 1 for t in range(inst.shape[0])])
    print(f'{uid}: {inst.shape}  ~{n:.0f} cells/frame')

## 5. Track + score each TEST movie

Per movie: compute the flow field once (grey = **mean** over that movie's channels), derive per-cell
`cell_flows` (`[u,v]` = first two feature dims) for position prediction, per-cell confetti
intensities for `w_color`, then Kalman + LAP with Mahalanobis + `w_flow` + `w_color`. Tracking runs
in µm (anisotropy is load-bearing). `w_color=1.0` is the current honest best; `w_flow=0.3` helps.

In [ ]:
scores_per_uid = {}
for uid in TEST_UIDS:
    instances_4d = np.asarray(instances_per_uid[uid])
    vol_uid      = np.asarray(volumes[uid][0])                          # [T, C, Z, H, W]
    ch           = CH_INDICES_MAP[uid]

    grey = vol_uid[:, ch].mean(axis=1).astype(np.float32)               # [T, Z, H, W]
    cell_feats, dense_flow_fields = compute_cell_flow_features(grey, instances_4d, n_workers=8)
    cell_flows = {t: {cid: f[:2] for cid, f in cells.items()}           # {t: {cid: [u, v]}}
                  for t, cells in cell_feats.items()}
    cell_intensities = extract_cell_intensities(instances_4d, vol_uid, ch)
    del grey, cell_feats; gc.collect()

    tracks = track_sequence(
        instances_4d, pix_res[uid],
        cell_flows=cell_flows,
        dense_flow_fields=dense_flow_fields,
        cell_intensities=cell_intensities,
        w_flow=0.3, w_color=1.0, color_ema=0.9,
        search_radius_um=10.0, max_cost=1.5,
        n_hist=5, momentum_decay=0.8, max_gap=1,
        process_noise=1.0, obs_noise=9.0, chi2_gate=9.21,
    )

    scores_per_uid[uid] = score_tracking(
        tracks, instances_4d, vol_uid,
        ch_indices=ch, pix_res=pix_res[uid], verbose=True)
    print(f'== {uid}: {len(tracks)} tracks ==')

## 6. Summary

`continuity` (↑ = less fragmentation) and `switch_rate` (↓ = fewer wrong assignments), on detected
cells only (no ghost-track inflation), across the held-out TEST movies. Beating both at once is the
open problem — see [`../docs/TRACKING.md`](../docs/TRACKING.md).

In [ ]:
for uid, s in scores_per_uid.items():
    cont = s.get('continuity', {}).get('mean')
    print(f"{uid}: continuity={cont:.3f}  switch_rate={s.get('color_switch_rate'):.3f}")